# Global settings

In [ ]:
import re
from typing import Optional

import numpy as np
import pandas as pd
from Bio import SeqIO

In [ ]:
def extract_uniprot_id(description: Optional[str]) -> Optional[str]:
    """
    Parse UniProt accession from FASTA description (Bio.SeqIO record.description).

    1) UniProt format: sp|ACCESSION|... or tr|ACCESSION|... (accession = first pipe field)
    2) Else first whitespace-delimited token, with optional leading '>' stripped

    Returns None only if description is empty or whitespace.
    """
    if not description or not str(description).strip():
        return None
    d = str(description).strip()
    m = re.match(r'^(?:sp|tr)\|([^|]+)\|', d)
    if m:
        return m.group(1)
    token = d.split()[0].lstrip('>')
    return token if token else None


def extract_uniprot_ids(input_fasta: str, output_file: str) -> int:
    """
    Extract UniProt IDs from FASTA header lines
    
    Args:
        input_fasta: Path to input FASTA file
        output_file: Path to output file (one UniProt ID per line)
    
    Returns:
        Number of UniProt IDs extracted
    """
    uniprot_ids = []
    for record in SeqIO.parse(input_fasta, "fasta"):
        uid = extract_uniprot_id(record.description)
        if uid:
            uniprot_ids.append(uid)
    
    with open(output_file, 'w') as f:
        for uniprot_id in uniprot_ids:
            f.write(uniprot_id + '\n')
    
    return len(uniprot_ids)

# Negative benchmark set construction

## Swiss-Prot download (candidate source)

In [ ]:
%%bash
wget -c https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.fasta.gz

In [ ]:
%%bash
gzip -d uniprot_sprot.fasta.gz
mv uniprot_sprot.fasta swissprot.fasta

## Length filter (50–3,000 amino acids)

In [ ]:
def filter_proteins_by_length(input_fasta, output_fasta, min_length=50, max_length=3000):
    """
    Filter proteins in a FASTA file by sequence length
    
    Args:
        input_fasta: Path to input FASTA file
        output_fasta: Path to output FASTA file
        min_length: Minimum sequence length (default 50)
        max_length: Maximum sequence length (default 3000)
    
    Returns:
        Tuple of (total_count, filtered_count) before and after filtering
    """
    total_count = 0
    filtered_count = 0
    filtered_records = []
    
    # Read and filter sequences
    for record in SeqIO.parse(input_fasta, "fasta"):
        total_count += 1
        seq_length = len(record.seq)
        
        if min_length <= seq_length <= max_length:
            filtered_count += 1
            filtered_records.append(record)
    
    # Write filtered sequences to output file
    if filtered_records:
        SeqIO.write(filtered_records, output_fasta, "fasta")
    
    return total_count, filtered_count

In [ ]:
input_file = "swissprot.fasta"  # Input FASTA
output_file = "swissprot_50-3000.fasta"  # Output FASTA

# Run filter
total, filtered = filter_proteins_by_length(input_file, output_file, min_length=50, max_length=3000)

# Print results
print(f"Proteins before filtering: {total}")
print(f"Proteins after filtering: {filtered}")
print(f"Proteins removed by filtering: {total - filtered}")
print(f"Fraction retained: {filtered/total*100:.2f}%")
print(f"\nFiltered sequences saved to: {output_file}")

## Homology filtering versus LLPS-focused databases and CD-CODE v1

_BLASTP-style criteria: sequence identity ≥40%, alignment coverage ≥70%._

In [ ]:
%%bash
# Remove DB homologs

set -euo pipefail

# Input files
POS=../data/potential_PSPs.fasta
NEG=swissprot_50-3000.fasta

# Thresholds (tune per task)
IDENTITY=40       # Percent identity threshold (e.g. ≥40% treated as homologous)
COVERAGE=0.7      # Coverage: alignment length / query or subject length ≥ 0.70
EVALUE=1e-10       # E-value threshold

# 1. Build positive-set database
diamond makedb --in $POS -d pos_db

# 2. Align negative sequences to positive database
diamond blastp \
    --query $NEG \
    --db pos_db \
    --out neg_vs_pos.tsv \
    --outfmt 6 qseqid sseqid pident length qlen slen evalue bitscore \
    --evalue $EVALUE \
    --max-target-seqs 1 \
    --threads 8

# 3. Filter by similarity
awk -v ID=$IDENTITY -v COV=$COVERAGE '
{
    qcov = $4 / $5;
    scov = $4 / $6;
    if ($3 >= ID && (qcov >= COV || scov >= COV)) {
        print $1;
    }
}' neg_vs_pos.tsv | sort -u > to_remove.list

# 4. Final negative set (remove high-homology hits)
seqkit grep -v -f to_remove.list $NEG > swissprot_50-3000_db_filtered.fasta

## Exclude proteins overlapping the assembled positive sequence set

In [ ]:
%%bash

POS=../data/potential_PSPs.fasta
IN=swissprot_50-3000_db_filtered.fasta
OUT=swissprot_50-3000_db_filtered_minus_pos.fasta

# 1) All sequence IDs from POS (first header field only, avoid space mismatch)
seqkit seq -n -i "$POS" | sort -u > pos_ids.list

# 2) Drop those IDs from IN
seqkit grep -v -f pos_ids.list "$IN" > "$OUT"

In [ ]:
%%bash
grep -c '>' swissprot.fasta
grep -c '>' swissprot_50-3000.fasta
grep -c '>' swissprot_50-3000_db_filtered.fasta
grep -c '>' swissprot_50-3000_db_filtered_minus_pos.fasta

## Remove training positives of evaluated predictors

_Droppler excluded when training IDs are unavailable._

In [ ]:
def filter_fasta(input_fasta: str, remove_ids, output_fasta: str) -> None:
    """Write FASTA keeping records whose UniProt ID is not in ``remove_ids``."""
    remove_ids = set(remove_ids)
    kept_records = []
    removed_count = 0
    total_count = 0

    for record in SeqIO.parse(input_fasta, "fasta"):
        total_count += 1
        uid = extract_uniprot_id(record.description)

        if uid not in remove_ids:
            kept_records.append(record)
        else:
            removed_count += 1

    SeqIO.write(kept_records, output_fasta, "fasta")

    print(f"Total sequences: {total_count}")
    print(f"Removed sequences: {removed_count}")
    print(f"Kept sequences: {len(kept_records)}")
    print(f"Output file: {output_fasta}")

In [ ]:
training_uids_df = pd.read_csv('../data/all_tools_training_uids.csv')
remove_ids = training_uids_df['uniprot'].tolist()

In [ ]:
input_fasta = "swissprot_50-3000_db_filtered_minus_pos.fasta"
output_fasta = "swissprot_50-3000_db_filtered_minus_pos_minus_training.fasta"

filter_fasta(input_fasta, remove_ids, output_fasta)

## Remove BioGRID v4.4 first-degree interactors of positive proteins

First-degree interactors from [BioGRID v4.4](https://thebiogrid.org/): pos_first-interactors.tsv.

In [ ]:
interactors = pd.read_csv('../data/pos_first-interactors.tsv', sep='\t', header=None)
remove_ids = interactors.iloc[:, 0].tolist()

In [ ]:
input_fasta = "swissprot_50-3000_db_filtered_minus_pos_minus_training.fasta"
output_fasta = "swissprot_50-3000_db_filtered_minus_pos_minus_training_minus_interactors.fasta"

filter_fasta(input_fasta, remove_ids, output_fasta)

## Exclude sequences with uncommon residues (e.g. B, J, O, U, X, Z)

In [ ]:
def find_non_standard_seqs(fasta_file: str) -> None:
    """List UniProt accessions whose sequence contains nonstandard amino acid letters."""
    # Set of 20 standard amino acids
    standard_aa = set("ACDEFGHIKLMNPQRSTVWY")
    
    non_standard_uids = []
    
    # Read FASTA
    for record in SeqIO.parse(fasta_file, "fasta"):
        # 1. Uppercase sequence (avoid case false positives)
        seq_str = str(record.seq).upper()
        seq_chars = set(seq_str)
        
        # 2. If sequence contains non-standard residues
        # (seq_chars - standard_aa) non-empty => non-standard
        if seq_chars - standard_aa:
            uid = extract_uniprot_id(record.description)
            if uid is None:
                uid = record.id
            non_standard_uids.append(uid)
            
    print(f"Done. Found {len(non_standard_uids)} sequences with non-standard amino acids.")
    
    return non_standard_uids


In [ ]:
remove_ids = find_non_standard_seqs('swissprot_50-3000_db_filtered_minus_pos_minus_training_minus_interactors.fasta')


In [ ]:
input_fasta = "swissprot_50-3000_db_filtered_minus_pos_minus_training_minus_interactors.fasta"
output_fasta = "swissprot_50-3000_db_filtered_minus_pos_minus_training_minus_interactors_removeNonstandard.fasta"

filter_fasta(input_fasta, remove_ids, output_fasta)

In [ ]:
input_fasta = "swissprot_50-3000_db_filtered_minus_pos_minus_training_minus_interactors_removeNonstandard.fasta"
output_file = "swissprot_50-3000_db_filtered_minus_pos_minus_training_minus_interactors_removeNonstandard.txt"

# Extract UniProt IDs
count = extract_uniprot_ids(input_fasta, output_file)

print(f"Successfully extracted {count} UniProt IDs")
print(f"UniProt IDs saved to: {output_file}")

## Taxonomic grouping

In [ ]:
def classify_species(taxonomic_lineage):
    """
    Classify organism from taxonomic lineage string
    
    Args:
        taxonomic_lineage (str): Taxonomic lineage string
    
    Returns:
        str: Class label
    """
    if not taxonomic_lineage or taxonomic_lineage == '':
        return 'Other'
    
    taxonomic_lineage = str(taxonomic_lineage).lower()
    
    # Check keywords in priority order
    if 'cellular organisms, bacteria' in taxonomic_lineage:
        return 'Bacteria'
    elif 'cellular organisms, archaea' in taxonomic_lineage:
        return 'Archaea'
    elif 'cellular organisms, eukaryota, opisthokonta, fungi' in taxonomic_lineage:
        return 'Fungi'
    elif 'amniota, mammalia' in taxonomic_lineage:
        return 'Mammals'
    elif 'cellular organisms, eukaryota, viridiplantae' in taxonomic_lineage:
        return 'Plants'
    elif 'cellular organisms, eukaryota, opisthokonta, metazoa' in taxonomic_lineage:
        return 'Non-mammalian animals'
    elif 'viruses' in taxonomic_lineage:
        return 'Viruses'
    else:
        return 'Other'
    
def process_row(row):
    taxonomic_lineage = str(row.get('Taxon', ''))
    species_class = classify_species(taxonomic_lineage)
    is_matched = taxonomic_lineage != ''
    return species_class, is_matched

UniProt ID mapping to get taxonomic lineage (local execution): swissprot_50-3000_db_filtered_minus_pos_minus_training_minus_interactors_removeNonstandard_idmapping.tsv

In [ ]:
all_data = pd.read_csv(f'swissprot_50-3000_db_filtered_minus_pos_minus_training_minus_interactors_removeNonstandard_idmapping.tsv',
                       sep='\t')
all_data['Taxon'] = all_data['Taxonomic lineage'].str.replace(r' \(.*?\)', '', regex=True).str.strip()

In [ ]:
# Apply row-wise
results = all_data.apply(process_row, axis=1, result_type='expand')
all_data['species_class'] = results[0]
all_data['is_matched'] = results[1]

# Counts
class_counts = all_data['species_class'].value_counts().to_dict()
matched_count = np.sum(all_data['is_matched'])

# Print summary
print("\nClassification counts:")
for class_name, count in sorted(class_counts.items()):
    print(f"  {class_name}: {count} records")

print(f"\nRecords with empty lineage: {len(all_data) - matched_count}")
print(f"Records with lineage: {matched_count}")
print(f"Match rate: {(matched_count / len(all_data) * 100):.2f}%")

In [ ]:
cols = ['From','Organism (ID)','species_class']
save_data = all_data[cols].copy()
save_data.columns = ['uniprot', 'Organism_ID', 'species_class']

# Mapping rules dict
mapping_rules = {
    'Archaea':'Prokaryotes',
    'Bacteria':'Prokaryotes',
    'Other': 'Protists',
    'Non-mammalian animals': 'Animals(NM)',
    'Mammals':'Animals(M)',
}

save_data['organism'] = save_data['species_class'].replace(mapping_rules)
save_data.to_csv('swissprot_50-3000_db_filtered_minus_pos_minus_training_minus_interactors_removeNonstandard_organism.csv', index=False)

## IDP versus non-IDP labeling

Residue-level disorder and binding scores from AIUPred pipeline (local execution): swissprot_50-3000_db_filtered_minus_pos_minus_training_minus_interactors_removeNonstandard_iupred_bindingScore.txt

In [ ]:
from __future__ import annotations
import matplotlib.pyplot as plt
import pandas as pd
import os
import sys
import re
import ast


_POSITIVE_FLAG = "1"
_NEGATIVE_FLAG = "0"


def dilate(states: str, max_length: int) -> str:
    """String dilation as in MobiDB-lite."""
    states = f"{_POSITIVE_FLAG * max_length}{states}{_POSITIVE_FLAG * max_length}"

    for level in range(1, max_length + 1):
        old = f"{_POSITIVE_FLAG * level}{_NEGATIVE_FLAG * level}{_POSITIVE_FLAG * level}"
        new = f"{_POSITIVE_FLAG * level}{_POSITIVE_FLAG * level}{_POSITIVE_FLAG * level}"
        for _ in range(level + 1):
            states = states.replace(old, new)

    return states[max_length:-max_length]


def erode(states: str, max_length: int) -> str:
    """String erosion as in MobiDB-lite."""
    states = f"{_NEGATIVE_FLAG * max_length}{states}{_NEGATIVE_FLAG * max_length}"

    for level in range(1, max_length + 1):
        old = f"{_NEGATIVE_FLAG * level}{_POSITIVE_FLAG * level}{_NEGATIVE_FLAG * level}"
        new = f"{_NEGATIVE_FLAG * level}{_NEGATIVE_FLAG * level}{_NEGATIVE_FLAG * level}"
        for _ in range(level + 1):
            states = states.replace(old, new)

    return states[max_length:-max_length]


def _repl_struct_by_disord(match: re.Match) -> str:
    """Replace matched long IDR + short gap + long IDR spans with '1'."""
    return _POSITIVE_FLAG * len(match.group(0))


def merge_long_disordered_regions(states: str) -> str:
    """Same as merge_long_disordered_regions in consensus.py."""
    pattern = r"{p}{{21,}}{n}{{1,10}}{p}{{21,}}".format(
        p=_POSITIVE_FLAG,
        n=_NEGATIVE_FLAG
    )

    while True:
        new_states = re.sub(
            pattern=pattern,
            repl=_repl_struct_by_disord,
            string=states
        )
        if new_states == states:
            return new_states
        states = new_states


def get_regions(states: str, min_length: int) -> list:
    """
    As get_regions in consensus.py:
    Return 0-based (start, end, flag) for flag != '0' and length >= min_length.
    """
    regions = []
    start = None
    current_flag = None

    for i, flag in enumerate(states):
        if flag != current_flag:
            if start is not None and current_flag != _NEGATIVE_FLAG:
                end = i - 1
                length = end - start + 1
                if length >= min_length:
                    regions.append((start, end, current_flag))
            start = i
            current_flag = flag

    if start is not None and current_flag != _NEGATIVE_FLAG:
        end = len(states) - 1
        length = end - start + 1
        if length >= min_length:
            regions.append((start, end, current_flag))

    return regions


def mobidb_lite_postproc_from_aiupred(scores, thr=0.5):
    """
    MobiDB-lite-style post-processing:
      1) score>=thr -> '1' / '0'
      2) dilate(max_length=3)
      3) erode(max_length=3)
      4) merge_long_disordered_regions
      5) get_regions(min_length=20), long IDRs only
    Returns: list of (start, end), inclusive 0-based
    """
    # 1) AIUPred scores -> initial state string
    states = "".join(
        _POSITIVE_FLAG if s >= thr else _NEGATIVE_FLAG
        for s in scores
    )

    # 2)–4) Morphology + gap merge (MobiDB-lite)
    states = dilate(states, max_length=3)
    states = erode(states, max_length=3)
    states = merge_long_disordered_regions(states)

    # 5) Extract long IDRs (>=20)
    regions = get_regions(states, min_length=20)
    return [(start, end) for (start, end, _) in regions]


def parse_aiupred_multi_fasta(path):
    proteins = []
    current_id = None
    current_scores = []

    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            if line.startswith("# "):
                continue

            if line.startswith("#>"):
                if current_id is not None:
                    proteins.append({
                        "uniprot": current_id,
                        "scores": current_scores,
                    })
                parts = line.lstrip("#>").split("|")
                if len(parts) >= 2:
                    current_id = parts[1]
                else:
                    current_id = line.lstrip("#>").strip()
                current_scores = []
            else:
                cols = line.split()
                if len(cols) < 3:
                    continue
                score = float(cols[2])
                current_scores.append(score)

    if current_id is not None:
        proteins.append({
            "uniprot": current_id,
            "scores": current_scores,
        })

    return proteins


def aiupred_to_long_idr_df_mobidb_like(path, thr=0.5):
    """
    Read AIUPred output -> MobiDB-lite postprocess for long IDRs ->
    Returns DataFrame:
        uniprot | regions_0based
        P12345  | "[(53, 78), (317, 349)]"
    """
    proteins = parse_aiupred_multi_fasta(path)

    records = []
    for prot in proteins:
        uid = prot["uniprot"]
        scores = prot["scores"]
        regions = mobidb_lite_postproc_from_aiupred(scores, thr=thr)
        records.append({
            "uniprot": uid,
            "regions_0based": str(regions),
        })

    return pd.DataFrame(records)

In [ ]:
data = aiupred_to_long_idr_df_mobidb_like('swissprot_50-3000_db_filtered_minus_pos_minus_training_minus_interactors_removeNonstandard_iupred_bindingScore.txt')

data["regions_list"] = data["regions_0based"].apply(ast.literal_eval)
data["IDP_type"] = data["regions_list"].apply(
    lambda lst: "IDP" if len(lst) > 0 else "non-IDP"
)

data.to_csv('swissprot_50-3000_db_filtered_minus_pos_minus_training_minus_interactors_removeNonstandard_iupred_idr_regions.tsv',
            sep='\t', index=False)

## CD-HIT within taxonomic groups (sequence identity cutoff 0.3)

### Define strata (taxonomy × IDP / non-IDP)

In [ ]:
IDR_df = pd.read_csv('swissprot_50-3000_db_filtered_minus_pos_minus_training_minus_interactors_removeNonstandard_iupred_idr_regions.tsv', sep='\t')
species_df = pd.read_csv('swissprot_50-3000_db_filtered_minus_pos_minus_training_minus_interactors_removeNonstandard_organism.csv')
class_df = pd.merge(species_df, IDR_df, on='uniprot')
class_df["class"] = class_df["organism"].astype(str) + "_" + class_df["IDP_type"].astype(str)
class_df.to_csv(
    'swissprot_50-3000_db_filtered_minus_pos_minus_training_minus_interactors_removeNonstandard_iupred_idr_regions_organism.tsv',
    sep='\t',
    index=False
)

### Run CD-HIT independently within each taxon

CD-HIT is run locally per taxon to reduce redundancy while preserving taxon-specific diversity: merged_after_cd-hit_by_species_30.fasta

In [ ]:
input_fasta = "merged_after_cd-hit_by_species_30.fasta"
output_file = "merged_after_cd-hit_by_species_uids_30.txt"

# Extract UniProt IDs
count = extract_uniprot_ids(input_fasta, output_file)

print(f"Successfully extracted {count} UniProt IDs")
print(f"UniProt IDs saved to: {output_file}")

# Show first 10 IDs as examples
with open(output_file, 'r') as f:
    first_10 = [line.strip() for line in f.readlines()[:10]]
    print(f"\nFirst 10 UniProt ID examples:")
    for i, uniprot_id in enumerate(first_10, 1):
        print(f"{i:2d}. {uniprot_id}")

## Final negative set: stratified sampling

_Target counts follow the taxonomic and IDP / non-IDP composition of the positive set (12,071 proteins in the final benchmark)._

### Compute per-stratum sample sizes from the positive composition

In [ ]:
class_df = pd.read_csv(
    'swissprot_50-3000_db_filtered_minus_pos_minus_training_minus_interactors_removeNonstandard_iupred_idr_regions_organism.tsv',
    sep='\t')

neg_ids = pd.read_csv('merged_after_cd-hit_by_species_uids_30.txt', header=None)[0].tolist()
neg_df = class_df[class_df['uniprot'].isin(neg_ids)]
neg_df.to_csv('negative_class.tsv', sep='\t', index=False)

Sample negatives proportionally to each class in the positive table
while maximizing total count under that constraint.


In [ ]:
from collections import Counter

import pandas as pd

def calculate_sampling_plan(positive_file, negative_file, output_file, target_total=None):
    """
    Compute stratified sampling plan
    
    Args:
        positive_file: Path to positive TSV
        negative_file: Path to negative TSV
        output_file: Path to output TSV
        target_total: Optional target total; if set, approach it while keeping class ratios
    """
    # Load positive table
    print(f"Loading positive table: {positive_file}")
    positive_df = pd.read_csv(positive_file, sep='\t')
    
    # Count per class in positive
    positive_counts = Counter(positive_df['class'])
    positive_total = len(positive_df)
    
    print(f"\nPositive set summary:")
    print(f"Total samples: {positive_total}")
    print(f"\nCounts per class:")
    for class_name, count in sorted(positive_counts.items()):
        proportion = count / positive_total
        print(f"  {class_name}: {count} ({proportion:.4f})")
    
    # Positive class proportions
    positive_proportions = {class_name: count / positive_total 
                           for class_name, count in positive_counts.items()}
    
    # Load negative table
    print(f"\nLoading negative table: {negative_file}")
    negative_df = pd.read_csv(negative_file, sep='\t')
    
    # Count per class in negative
    negative_counts = Counter(negative_df['class'])
    negative_total = len(negative_df)
    
    print(f"\nNegative set summary:")
    print(f"Total samples: {negative_total}")
    print(f"\nCounts per class:")
    for class_name, count in sorted(negative_counts.items()):
        print(f"  {class_name}: {count}")
    
    # Check negative has all positive classes
    missing_classes = set(positive_proportions.keys()) - set(negative_counts.keys())
    if missing_classes:
        print(f"\nWarning: negative missing classes: {missing_classes}")
        # Drop missing classes from proportions
        for class_name in missing_classes:
            del positive_proportions[class_name]
    
    # Max multiplier k s.t. k * p_c <= negative_count for each class
    # k = min(negative_count / positive_proportion) for all classes
    k_values = []
    limiting_class = None
    for class_name, proportion in positive_proportions.items():
        if class_name in negative_counts:
            k = negative_counts[class_name] / proportion
            k_values.append(k)
    
    # Limiting class (smallest k)
    max_k = min(k_values)
    for class_name, proportion in positive_proportions.items():
        if class_name in negative_counts:
            k = negative_counts[class_name] / proportion
            if abs(k - max_k) < 0.01:  # float tolerance
                limiting_class = class_name
                break
    
    if not k_values:
        print("Error: no valid sampling plan")
        return
    
    if limiting_class:
        print(f"\nLimiting class: {limiting_class} (available: {negative_counts[limiting_class]}, "
              f"required proportion: {positive_proportions[limiting_class]:.4f})")
    print(f"Max multiplier k = {max_k:.2f}")
    
    # If target_total set, derive k
    if target_total is not None:
        # Sum of proportions (should be 1)
        total_proportion = sum(positive_proportions.values())
        # target_total = k * sum(p)
        target_k = target_total / total_proportion if total_proportion > 0 else max_k
        # final_k = min(target_k, max_k)
        final_k = min(target_k, max_k)
        print(f"\nTarget total: {target_total}")
        print(f"k from target total: {target_k:.2f}")
        print(f"Final k: {final_k:.2f} (capped by max k: {max_k:.2f})")
    else:
        final_k = max_k
        print(f"\nNo target total; using max k = {final_k:.2f}")
    
    # Per-class sample counts
    sampling_plan = []
    total_sampled = 0
    
    for class_name, proportion in sorted(positive_proportions.items()):
        if class_name in negative_counts:
            # Ideal count from final_k
            ideal_count = final_k * proportion
            
            # Actual count (floor)
            sample_count = int(ideal_count)
            
            # Cap by availability
            sample_count = min(sample_count, negative_counts[class_name])
            
            # Utilization
            actual_proportion = sample_count / ideal_count if ideal_count > 0 else 0
            
            sampling_plan.append({
                'class': class_name,
                'positive_count': positive_counts.get(class_name, 0),
                'positive_proportion': proportion,
                'negative_available': negative_counts[class_name],
                'sample_count': sample_count,
                'ideal_count': ideal_count,
                'utilization_rate': actual_proportion
            })
            
            total_sampled += sample_count
    
    # Build result DataFrame
    result_df = pd.DataFrame(sampling_plan)
    
    # Save
    result_df.to_csv(output_file, sep='\t', index=False)
    print(f"\nSampling plan saved to: {output_file}")
    
    # Summary table
    print(f"\nSampling plan summary:")
    print(f"{'Class':<30} {'Pos count':<15} {'Pos prop':<15} {'Neg avail':<15} {'Sample':<15} {'Ideal':<15} {'Util':<10}")
    print("-" * 120)
    
    for _, row in result_df.iterrows():
        print(f"{row['class']:<30} {row['positive_count']:<15} {row['positive_proportion']:<15.4f} "
              f"{row['negative_available']:<15} {row['sample_count']:<15} "
              f"{row['ideal_count']:<15.2f} {row['utilization_rate']:<10.2%}")
    
    print(f"\nTotal sampled: {total_sampled}")
    print(f"Expected total (at max k): {sum(row['ideal_count'] for _, row in result_df.iterrows()):.2f}")
    
    # Actual vs target proportions
    print(f"\nActual proportions after sampling:")
    for _, row in result_df.iterrows():
        if total_sampled > 0:
            actual_prop = row['sample_count'] / total_sampled
            target_prop = row['positive_proportion']
            diff = abs(actual_prop - target_prop)
            print(f"  {row['class']:<30} target: {target_prop:.4f}  actual: {actual_prop:.4f}  diff: {diff:.4f}")
    

In [ ]:
calculate_sampling_plan(
    positive_file='positive_class.tsv', 
    negative_file='negative_class.tsv', 
    output_file='negative_sampling_plan_30.tsv'
)

### ESM-2 mean-pooled sequence embeddings

Compute ESM2 embeddings for all sequences in a FASTA file
- Each sequence saved as .npy (seq_len × 1280)
- Mean-pooled vectors saved to .pkl (dict: UniProt id -> 1280-d vector)


In [ ]:
import os
import pickle
import sys
from typing import Optional

import numpy as np
import torch
from Bio import SeqIO
from tqdm import tqdm

# Requires ``extract_uniprot_id`` from the utilities cell (run that cell first).


def compute_esm2_embedding(
    sequence: str, model, alphabet, batch_converter, torch_device: torch.device
):
    """
    Compute ESM2 embedding for one sequence.

    Args:
        sequence: Amino acid sequence string
        model: ESM2 model
        alphabet: ESM2 alphabet
        batch_converter: ESM batch converter
        torch_device: Target ``torch.device`` (CUDA or CPU)

    Returns:
        Per-residue embedding matrix of shape (seq_len, 1280).
    """
    data = [(None, sequence)]
    _batch_labels, _batch_strs, batch_tokens = batch_converter(data)
    batch_tokens = batch_tokens.to(torch_device)

    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[33])
        token_representations = results["representations"][33]
        sequence_representation = token_representations[0, 1:-1, :].cpu().numpy()

    return sequence_representation


def process_fasta_file(
    input_fasta: str,
    output_dir: str,
    npy_dir: Optional[str] = None,
    pkl_file: Optional[str] = None,
    device: Optional[int] = None,
):
    """
    Process FASTA: ESM2 embeddings for all sequences
    
    Args:
        input_fasta: Path to input FASTA file
        output_dir: Output directory
        npy_dir: .npy dir (default: output_dir/npy)
        pkl_file: .pkl path (default: output_dir/embeddings.pkl)
        device: CUDA device index (int); use None for cuda:0 when CUDA is available.
    """
    # Create dirs
    os.makedirs(output_dir, exist_ok=True)
    
    # Paths for npy/pkl
    if npy_dir is None:
        npy_dir = os.path.join(output_dir, "npy")
    if pkl_file is None:
        pkl_file = os.path.join(output_dir, "embeddings.pkl")
    
    os.makedirs(npy_dir, exist_ok=True)
    
    if torch.cuda.is_available():
        cuda_index = 0 if device is None else int(device)
        torch_device = torch.device(f"cuda:{cuda_index}")
    else:
        torch_device = torch.device("cpu")
    
    # Load ESM2
    print("Loading ESM2 model...")
    try:
        import esm
        model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
        batch_converter = alphabet.get_batch_converter()
        model.eval()
        model = model.to(torch_device)
        print("ESM2 model loaded")
    except ImportError:
        print("Error: install esm: pip install fair-esm")
        sys.exit(1)
    except Exception as e:
        print(f"Error: failed to load ESM2: {e}")
        sys.exit(1)
    
    # Read FASTA and collect sequences
    print(f"Reading FASTA: {input_fasta}")
    sequences = []
    uniprot_ids = []
    
    for record in SeqIO.parse(input_fasta, "fasta"):
        uniprot_id = extract_uniprot_id(record.description)
        if uniprot_id is None:
            print(f"Warning: could not parse UniProt ID, skipping: {record.description[:50]}")
            continue
        
        sequences.append((uniprot_id, str(record.seq)))
        uniprot_ids.append(uniprot_id)
    
    total = len(sequences)
    print(f"Found {total} valid sequences")
    
    # Mean-pooled embeddings dict
    embeddings_dict = {}
    
    # Per-sequence loop
    print("Computing embeddings...")
    failed_count = 0
    
    for i, (uniprot_id, sequence) in enumerate(tqdm(sequences, desc="Sequences")):
        try:
            npy_path = os.path.join(npy_dir, f"{uniprot_id}.npy")
            
            # Embedding
            embedding_matrix = compute_esm2_embedding(
                sequence, model, alphabet, batch_converter, torch_device
            )
            
            # Save .npy
            np.save(npy_path, embedding_matrix)
            
            # Mean pool
            mean_embedding = np.mean(embedding_matrix, axis=0)
            # Store as list
            embeddings_dict[uniprot_id] = mean_embedding.tolist()
            
        except Exception as e:
            print(f"\nError on {uniprot_id}: {e}")
            failed_count += 1
            continue
        
    
    # Final pkl write
    print(f"\nSaving embeddings to: {pkl_file}")
    with open(pkl_file, 'wb') as f:
        pickle.dump(embeddings_dict, f)
    
    print(f"\nDone.")
    print(f"  - Total sequences: {total}")
    print(f"  - Succeeded: {total - failed_count}")
    print(f"  - Failed: {failed_count}")
    print(f"  - .npy files in: {npy_dir}")

In [ ]:
process_fasta_file(
    input_fasta='merged_after_cd-hit_by_species_30.fasta',
    output_dir='esm2_output',
    device=0,
    pkl_file='merged_after_cd-hit_by_species_30_esm.pkl'
)

### Within-stratum diversity selection (k-center greedy, cosine distance)

Diverse class-wise sampler for protein sequences using ESM-2 mean-pooled embeddings.

In [ ]:
import pickle
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd


def _find_col(df: pd.DataFrame, candidates: List[str]) -> str:
    """Return the first matching column name (case-insensitive, strip) from candidates. Raise if not found."""
    lower_map = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        key = cand.lower().strip()
        if key in lower_map:
            return lower_map[key]
    # fuzzy: try contains
    for c in df.columns:
        cl = c.lower().strip()
        if any(key in cl for key in [x.lower().strip() for x in candidates]):
            return c
    raise KeyError(f"Could not find any of columns {candidates} in {list(df.columns)}")


def load_embeddings(pkl_path: str) -> Dict[str, np.ndarray]:
    with open(pkl_path, "rb") as f:
        obj = pickle.load(f)
    emb = {}
    for k, v in obj.items():
        arr = np.asarray(v, dtype=np.float32)
        if arr.ndim != 1:
            raise ValueError(f"Embedding for {k} is not 1D (got shape {arr.shape})")
        emb[str(k)] = arr
    return emb


def l2_normalize_matrix(X: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return X / norms


def kcenter_greedy_cosine(X_norm: np.ndarray, k: int, rng: np.random.Generator) -> np.ndarray:
    """K-center greedy under cosine distance (1 - dot). X must be L2-normalized rows. Returns indices into X."""
    n = X_norm.shape[0]
    if k >= n:
        return np.arange(n, dtype=int)
    # Heuristic init: pick a random index, then farthest-first
    first = int(rng.integers(0, n))
    selected = [first]
    # min distance to the selected set, init with distance to first
    min_dist = 1.0 - (X_norm @ X_norm[first])
    for _ in range(1, k):
        # pick the point with the largest min distance to current selected set
        i = int(np.argmax(min_dist))
        selected.append(i)
        # update min_dist with distances to the newly added point
        new_d = 1.0 - (X_norm @ X_norm[i])
        min_dist = np.minimum(min_dist, new_d)
    return np.array(selected, dtype=int)


def per_class_select(ids: List[str],
                     embeddings: Dict[str, np.ndarray],
                     k: int,
                     seed: int = 42) -> Tuple[List[str], Dict]:
    """Return selected ids and a stats dict."""
    # Filter ids that exist in embeddings
    ids2 = [i for i in ids if i in embeddings]
    n_missing = len(ids) - len(ids2)
    if len(ids2) == 0:
        return [], {"available": 0, "requested": k, "selected": 0, "missing_in_embeddings": n_missing}

    X = np.stack([embeddings[i] for i in ids2], axis=0)
    X = l2_normalize_matrix(X)

    rng = np.random.default_rng(seed)
    k_eff = min(k, X.shape[0])
    sel_idx = kcenter_greedy_cosine(X, k_eff, rng)
    selected_ids = [ids2[i] for i in sel_idx]

    stats = {
        "available": int(X.shape[0]),
        "requested": int(k),
        "selected": int(k_eff),
        "missing_in_embeddings": int(n_missing),
    }
    if k_eff < k:
        stats["warning"] = f"requested {k} but only {X.shape[0]} available; selected all available."
    return selected_ids, stats

In [ ]:
embeddings = 'merged_after_cd-hit_by_species_30_esm.pkl'
class_map = 'negative_class.tsv'
plan = 'negative_sampling_plan_30.tsv'
output = 'negative_selection.tsv'
seed = 42

# Load inputs
print(f"Loading embeddings from: {embeddings}")
emb = load_embeddings(embeddings)
print(f"Embeddings loaded: {len(emb)} sequences.")

print(f"Loading class map from: {class_map}")
df_map = pd.read_csv(class_map, sep="\t", dtype=str)
id_col = _find_col(df_map, ["uniprot", "uniprot_id", 'uid'])
class_col = _find_col(df_map, ["class", "label", "category"])

# Normalize values
df_map[id_col] = df_map[id_col].astype(str)
df_map[class_col] = df_map[class_col].astype(str)

print(f"Loading plan from: {plan}")
df_plan = pd.read_csv(plan, sep="\t", dtype=str)
plan_class_col = _find_col(df_plan, ["class", "label", "category"])
plan_count_col = _find_col(df_plan, ["sample_count", "n_sample", "count"])

# Coerce types
df_plan[plan_count_col] = pd.to_numeric(df_plan[plan_count_col], errors="coerce").fillna(0).astype(int)

# Build per-class id lists
ids_by_class = (df_map[[id_col, class_col]]
                .dropna()
                .drop_duplicates()
                .groupby(class_col)[id_col]
                .apply(list)
                .to_dict())

# Run per-class selection
selections = []
stats = {}
for _, row in df_plan.iterrows():
    c = str(row[plan_class_col])
    k = int(row[plan_count_col])
    ids = ids_by_class.get(c, [])
    sel_ids, st = per_class_select(ids, emb, k, seed=seed)
    stats[c] = st
    for u in sel_ids:
        selections.append((c, u))

df_out = pd.DataFrame(selections, columns=["class", "uniprot_id"])
df_out.to_csv(output, sep="\t", index=False)
print(f"Wrote selections: {output}")